In [1]:
import pandas as pd
import numpy as np
import joblib
import os

model = joblib.load("../models/trained/cms_acceptance_model.joblib")
df = pd.read_csv("../data/interim/cms_suppliers_cleaned.csv")

print("Model loaded:", type(model).__name__)
print("Providers to score:", len(df))

Model loaded: HistGradientBoostingClassifier
Providers to score: 57197


In [2]:
df["participationbegindate_parsed"] = pd.to_datetime(df["participationbegindate"], errors="coerce")

reference_date = df["participationbegindate_parsed"].max()
df["tenure_days"] = (reference_date - df["participationbegindate_parsed"]).dt.days
df["is_suspected_placeholder_date"] = (df["date_reliability_flag"] == "Suspected placeholder date").astype(int)

print("tenure_days and is_suspected_placeholder_date recreated.")
print(df[["tenure_days", "is_suspected_placeholder_date"]].describe())


tenure_days and is_suspected_placeholder_date recreated.
        tenure_days  is_suspected_placeholder_date
count  57197.000000                   57197.000000
mean    4592.754795                       0.095617
std     3704.558274                       0.294068
min        0.000000                       0.000000
25%     1583.000000                       0.000000
50%     3493.000000                       0.000000
75%     7415.000000                       0.000000
max    31592.000000                       1.000000


In [3]:
specialty_dummies = df["specialitieslist"].str.get_dummies(sep="|").add_prefix("specialty_")
supply_dummies = df["supplieslist"].str.get_dummies(sep="|").add_prefix("supply_")
state_dummies = pd.get_dummies(df["practicestate"], prefix="state")

feature_cols_numeric = ["latitude", "longitude", "tenure_days", "is_suspected_placeholder_date"]

X_full = pd.concat([
    df[feature_cols_numeric],
    state_dummies,
    specialty_dummies,
    supply_dummies
], axis=1)

print("Full feature matrix shape:", X_full.shape)

# Critical check: must exactly match the 166 columns the model was trained on
model_features = model.feature_names_in_ if hasattr(model, "feature_names_in_") else None
if model_features is not None:
    missing_in_full = set(model_features) - set(X_full.columns)
    extra_in_full = set(X_full.columns) - set(model_features)
    print("Columns model expects but missing here:", missing_in_full)
    print("Extra columns here not seen by model:", extra_in_full)
else:
    print("Model does not expose feature_names_in_ - will verify by column count instead.")
    print("Expected 166 columns, got:", X_full.shape[1])

Full feature matrix shape: (57197, 166)
Columns model expects but missing here: set()
Extra columns here not seen by model: set()


In [4]:
predicted_proba = model.predict_proba(X_full)[:, 1]
predicted_class = (predicted_proba >= 0.5).astype(int)

df["Predicted_Probability"] = predicted_proba
df["Predicted_Class"] = predicted_class
df["Predicted_Class_Label"] = df["Predicted_Class"].map({1: "Accepting", 0: "Not Accepting"})

# Risk level based on predicted probability - explicit thresholds, documented
def assign_risk_level(prob):
    if prob < 0.3:
        return "High Risk"
    elif prob < 0.6:
        return "Medium Risk"
    else:
        return "Low Risk"

df["Risk_Level"] = df["Predicted_Probability"].apply(assign_risk_level)

df["Correct_Prediction"] = (df["Predicted_Class"] == df["acceptsassignement_bool"].astype(int)).astype(int)
df["Prediction_Confidence"] = np.abs(df["Predicted_Probability"] - 0.5) * 2  # 0 = uncertain, 1 = very confident

df["Model_Version"] = "HistGradientBoostingClassifier_v1"
df["Scoring_Date"] = pd.Timestamp.now().strftime("%Y-%m-%d")

print("Predictions generated for all providers.")
print()
print("Predicted_Class distribution:")
print(df["Predicted_Class_Label"].value_counts())
print()
print("Risk_Level distribution:")
print(df["Risk_Level"].value_counts())
print()
print("Overall prediction accuracy (full dataset, includes training data - not a true test metric):")
print(round(df["Correct_Prediction"].mean() * 100, 2), "%")

Predictions generated for all providers.

Predicted_Class distribution:
Predicted_Class_Label
Accepting        30346
Not Accepting    26851
Name: count, dtype: int64

Risk_Level distribution:
Risk_Level
Low Risk       27992
High Risk      22620
Medium Risk     6585
Name: count, dtype: int64

Overall prediction accuracy (full dataset, includes training data - not a true test metric):
90.88 %


In [5]:
powerbi_export = df[[
    "provider_id",
    "practicecity",
    "practicestate",
    "latitude",
    "longitude",
    "participationbegindate",
    "date_reliability_flag",
    "specialitieslist",
    "supplieslist",
    "acceptsassignement",
    "acceptsassignement_bool",
    "Predicted_Probability",
    "Predicted_Class",
    "Predicted_Class_Label",
    "Prediction_Confidence",
    "Risk_Level",
    "Correct_Prediction",
    "Model_Version",
    "Scoring_Date",
]].copy()

powerbi_export = powerbi_export.rename(columns={
    "acceptsassignement": "Actual_Acceptance",
    "acceptsassignement_bool": "Actual_Acceptance_Bool",
    "practicecity": "City",
    "practicestate": "State",
    "participationbegindate": "ParticipationBeginDate",
    "specialitieslist": "SpecialitiesList",
    "supplieslist": "SuppliesList",
})

print("Export shape:", powerbi_export.shape)
print()
print("Grain: one row per provider_id (57,197 distinct providers, verified no duplicates in Notebook 01)")
print()
powerbi_export.head(3)

Export shape: (57197, 19)

Grain: one row per provider_id (57,197 distinct providers, verified no duplicates in Notebook 01)



,provider_id,City,State,latitude,longitude,ParticipationBeginDate,date_reliability_flag,SpecialitiesList,SuppliesList,Actual_Acceptance,Actual_Acceptance_Bool,Predicted_Probability,Predicted_Class,Predicted_Class_Label,Prediction_Confidence,Risk_Level,Correct_Prediction,Model_Version,Scoring_Date
0,34363333,IRVINGTON,NJ,40.735948,-74.215433,2026-07-01,Likely genuine date,Pharmacy,Epoetin|Immunosuppressive Drugs|Infusion Drugs...,True,True,0.843917,1,Accepting,0.687834,Low Risk,1,HistGradientBoostingClassifier_v1,2026-07-17
1,34363327,PHILADELPHIA,PA,39.947301,-75.166342,2026-07-01,Likely genuine date,Pharmacy,Nebulizer Drugs,False,False,0.800875,1,Accepting,0.601750,Low Risk,0,HistGradientBoostingClassifier_v1,2026-07-17
2,34363291,WHARTON,NJ,40.899634,-74.582437,2026-06-15,Likely genuine date,Pharmacy,Epoetin|Immunosuppressive Drugs|Infusion Drugs...,True,True,0.827271,1,Accepting,0.654542,Low Risk,1,HistGradientBoostingClassifier_v1,2026-07-17


In [6]:
os.makedirs("../data/processed", exist_ok=True)

powerbi_export.to_csv("../data/processed/cms_supplier_predictions.csv", index=False)

print("Exported: data/processed/cms_supplier_predictions.csv")
print()
print("=== GRAIN DEFINITION ===")
print("One row per provider_id.")
print(f"Total rows: {len(powerbi_export)}")
print(f"Distinct provider_id: {powerbi_export['provider_id'].nunique()}")
print(f"Grain confirmed: {'PASS' if len(powerbi_export) == powerbi_export['provider_id'].nunique() else 'FAIL - DUPLICATE GRAIN'}")
print()
print("=== SCHEMA NOTES FOR POWER BI ===")
print("- SpecialitiesList and SuppliesList are pipe-delimited multi-value fields.")
print("  These require bridge tables (BridgeProviderSpecialty, BridgeProviderSupply) in Power BI")
print("  to avoid duplicating fact rows. Do NOT split this table directly - it will break the")
print("  one-row-per-provider grain that KPIs like Total Providers depend on.")
print("- Predicted_Probability is a continuous 0-1 value, distinct from Predicted_Class (binary)")
print("  and Risk_Level (3-category). Do not conflate these three fields in DAX measures.")
print("- 'Actual_Acceptance_Bool' reflects real observed behavior; 'Predicted_Class' and")
print("  'Predicted_Probability' are model outputs and must be visually/labelly distinguished")
print("  from actual values throughout the dashboard.")


Exported: data/processed/cms_supplier_predictions.csv

=== GRAIN DEFINITION ===
One row per provider_id.
Total rows: 57197
Distinct provider_id: 57197
Grain confirmed: PASS

=== SCHEMA NOTES FOR POWER BI ===
- SpecialitiesList and SuppliesList are pipe-delimited multi-value fields.
  These require bridge tables (BridgeProviderSpecialty, BridgeProviderSupply) in Power BI
  to avoid duplicating fact rows. Do NOT split this table directly - it will break the
  one-row-per-provider grain that KPIs like Total Providers depend on.
- Predicted_Probability is a continuous 0-1 value, distinct from Predicted_Class (binary)
  and Risk_Level (3-category). Do not conflate these three fields in DAX measures.
- 'Actual_Acceptance_Bool' reflects real observed behavior; 'Predicted_Class' and
  'Predicted_Probability' are model outputs and must be visually/labelly distinguished
  from actual values throughout the dashboard.
